In [ ]:
!pip install -q yfinance ta

  Preparing metadata (setup.py) ... done


In [ ]:
import yfinance as yf
import pandas as pd
import ta

In [ ]:
symbols = [
    "GC=F",      # Gold
    "DX-Y.NYB",  # Dollar Index
    "^GSPC",     # S&P500
    "CL=F",      # Oil
    "^VIX"       # VIX
]

df = yf.download(
    symbols,
    start="2000-08-31",
    end="2026-08-10"
)

/tmp/ipykernel_1872/531016450.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  5 of 5 completed


In [ ]:
gold = pd.DataFrame(index=df.index)

gold["Gold_Open"]   = df["Open"]["GC=F"]
gold["Gold_High"]   = df["High"]["GC=F"]
gold["Gold_Low"]    = df["Low"]["GC=F"]
gold["Gold_Close"]  = df["Close"]["GC=F"]
gold["Gold_Volume"] = df["Volume"]["GC=F"]

gold["DXY"]    = df["Close"]["DX-Y.NYB"]
gold["SP500"]  = df["Close"]["^GSPC"]
gold["Oil"]    = df["Close"]["CL=F"]
gold["VIX"]    = df["Close"]["^VIX"]

In [ ]:
gold.head()

,Gold_Open,Gold_High,Gold_Low,Gold_Close,Gold_Volume,DXY,SP500,Oil,VIX
Date,,,,,,,,,
2000-08-31,274.799988,278.299988,274.799988,278.299988,0.0,112.599998,1517.680054,33.099998,16.840000
2000-09-01,277.000000,277.000000,277.000000,277.000000,0.0,111.419998,1520.770020,33.380001,17.530001
2000-09-05,275.799988,275.799988,275.799988,275.799988,2.0,112.410004,1507.079956,33.799999,19.820000
2000-09-06,274.200012,274.200012,274.200012,274.200012,0.0,114.120003,1492.250000,34.950001,20.790001
2000-09-07,274.000000,274.000000,274.000000,274.000000,125.0,113.650002,1502.510010,35.330002,19.420000


In [ ]:
gold.isna().sum()

,0
Gold_Open,46
Gold_High,46
Gold_Low,46
Gold_Close,46
Gold_Volume,46
DXY,6
SP500,33
Oil,42
VIX,32


In [ ]:
gold = gold.dropna(subset=["Gold_Open","Gold_High","Gold_Low","Gold_Close"])

gold[["DXY","SP500","Oil","VIX"]] = (gold[["DXY","SP500","Oil","VIX"]].ffill())

### Feature Engineering

In [ ]:
# Lag Features
gold["Lag1"] = gold["Gold_Close"].shift(1)
gold["Lag7"] = gold["Gold_Close"].shift(7)

In [ ]:
# Moving Averages
gold["MA7"] = gold["Gold_Close"].rolling(7).mean()
gold["MA30"] = gold["Gold_Close"].rolling(30).mean()

In [ ]:
# Exponential Moving Average
gold["EMA20"] = gold["Gold_Close"].ewm(span=20, adjust=False).mean()

In [ ]:
# Daily Return
gold["Daily_Return"] = gold["Gold_Close"].pct_change()

In [ ]:
gold.head()

,Gold_Open,Gold_High,Gold_Low,Gold_Close,Gold_Volume,DXY,SP500,Oil,VIX,Lag1,Lag7,MA7,MA30,EMA20,Daily_Return
Date,,,,,,,,,,,,,,,
2000-08-31,274.799988,278.299988,274.799988,278.299988,0.0,112.599998,1517.680054,33.099998,16.840000,NaN,NaN,NaN,NaN,278.299988,NaN
2000-09-01,277.000000,277.000000,277.000000,277.000000,0.0,111.419998,1520.770020,33.380001,17.530001,278.299988,NaN,NaN,NaN,278.176179,-0.004671
2000-09-05,275.799988,275.799988,275.799988,275.799988,2.0,112.410004,1507.079956,33.799999,19.820000,277.000000,NaN,NaN,NaN,277.949875,-0.004332
2000-09-06,274.200012,274.200012,274.200012,274.200012,0.0,114.120003,1492.250000,34.950001,20.790001,275.799988,NaN,NaN,NaN,277.592746,-0.005801
2000-09-07,274.000000,274.000000,274.000000,274.000000,125.0,113.650002,1502.510010,35.330002,19.420000,274.200012,NaN,NaN,NaN,277.250579,-0.000729


### Technical Indicators

In [ ]:
# RSI
gold["RSI"] = ta.momentum.RSIIndicator(
    close=gold["Gold_Close"],
    window=14
).rsi()

In [ ]:
# MACD
macd = ta.trend.MACD(close=gold["Gold_Close"])

gold["MACD"] = macd.macd()
gold["MACD_Signal"] = macd.macd_signal()
gold["MACD_Hist"] = macd.macd_diff()

In [ ]:
# Bollinger Bands
bb = ta.volatility.BollingerBands(
    close=gold["Gold_Close"],
    window=20,
    window_dev=2
)

gold["BB_Upper"] = bb.bollinger_hband()
gold["BB_Middle"] = bb.bollinger_mavg()
gold["BB_Lower"] = bb.bollinger_lband()

In [ ]:
# ATR
gold["ATR"] = ta.volatility.AverageTrueRange(
    high = gold["Gold_High"],
    low = gold["Gold_Low"],
    close = gold["Gold_Close"],
    window = 14
).average_true_range()

In [ ]:
# Target
gold["Target"] = gold["Gold_Close"].shift(-1)

In [ ]:
(gold.isnull().sum())[gold.isnull().sum()>0]

,0
Lag1,1
Lag7,7
MA7,6
MA30,29
Daily_Return,1
RSI,13
MACD,25
MACD_Signal,33
MACD_Hist,33
BB_Upper,19


In [ ]:
gold = gold.dropna()

In [ ]:
gold.head()

,Gold_Open,Gold_High,Gold_Low,Gold_Close,Gold_Volume,DXY,SP500,Oil,VIX,Lag1,...,Daily_Return,RSI,MACD,MACD_Signal,MACD_Hist,BB_Upper,BB_Middle,BB_Lower,ATR,Target
Date,,,,,,,,,,,,,,,,,,,,,
2000-10-18,270.299988,270.299988,270.299988,270.299988,0.0,117.330002,1342.130005,33.549999,28.719999,271.100006,...,-0.002951,44.027677,-0.811933,-0.890278,0.078346,277.007532,272.364998,267.722464,1.797421,270.100006
2000-10-19,270.100006,270.100006,270.100006,270.100006,1.0,116.940002,1388.760010,32.950001,25.090000,270.299988,...,-0.000740,43.589968,-0.887787,-0.889780,0.001993,277.016104,272.354999,267.693893,1.683319,271.200012
2000-10-20,271.200012,271.200012,271.200012,271.200012,32.0,117.160004,1396.930054,34.299999,24.240000,270.100006,...,0.004073,46.727258,-0.849350,-0.881694,0.032344,277.007681,272.325000,267.642319,1.641653,270.100006
2000-10-23,271.700012,271.700012,270.100006,270.100006,1.0,117.540001,1395.780029,33.730000,24.690001,271.200012,...,-0.004056,44.086729,-0.897306,-0.884817,-0.012490,276.828986,272.125000,267.421014,1.638679,270.100006
2000-10-24,269.799988,270.100006,269.799988,270.100006,16.0,117.419998,1398.130005,33.369999,24.280001,270.100006,...,0.000000,44.086729,-0.924653,-0.892784,-0.031869,276.643831,271.935001,267.226170,1.543060,266.200012


In [ ]:
gold[['Gold_Close', 'Lag1', 'Target']].head()

,Gold_Close,Lag1,Target
Date,,,
2000-10-18,270.299988,271.100006,270.100006
2000-10-19,270.100006,270.299988,271.200012
2000-10-20,271.200012,270.100006,270.100006
2000-10-23,270.100006,271.200012,270.100006
2000-10-24,270.100006,270.100006,266.200012


In [ ]:
gold.columns

Index(['Gold_Open', 'Gold_High', 'Gold_Low', 'Gold_Close', 'Gold_Volume',
       'DXY', 'SP500', 'Oil', 'VIX', 'Lag1', 'Lag7', 'MA7', 'MA30', 'EMA20',
       'Daily_Return', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper',
       'BB_Middle', 'BB_Lower', 'ATR', 'Target'],
      dtype='object')

In [ ]:
gold.shape

(6474, 24)

In [ ]:
gold.iloc[0]

,2000-10-18
Gold_Open,270.299988
Gold_High,270.299988
Gold_Low,270.299988
Gold_Close,270.299988
Gold_Volume,0.000000
DXY,117.330002
SP500,1342.130005
Oil,33.549999
VIX,28.719999
Lag1,271.100006


In [ ]:
gold.to_csv("gold_data.csv")